# Break the Model 🔨

Every challenge gives you a **working model**. Your mission: **make it fail** on purpose – then **fix it**.
Breaking things is the fastest way to understand *why* the good practices exist.

**How it works**
1. Read the story and run the working model.
2. Do the 💥 *Break it* task: change the marked variables / fill in the code.
3. Run the ✅ check cell – it tells you whether you succeeded.
4. Do the 🛠️ *Fix it* task.
5. Stuck? Open the hidden solution (click the ▶ *Solution* line).

| # | Challenge | Concept | Note |
|---|---|---|---|
| 1 | The polynomial that memorised | overfitting | [[Overfitting and Regularization]] |
| 2 | 90 % accuracy on pure noise | data leakage in CV | [[Cross-Validation and Model Selection]] |
| 3 | The suspiciously perfect feature | target leakage | [[Common Pitfalls]] |
| 4 | The 99 % accurate fraud detector | class imbalance | [[Model Evaluation and Metrics]] |
| 5 | Fool the digit classifier | adversarial examples | [[Logistic Regression]] |
| 6 | It worked in the lab… | distribution shift | [[MLOps Overview]] |
| 7 | The scale that broke kNN | feature scaling | [[k-Nearest Neighbors]] |


In [ ]:
import numpy as np, matplotlib.pyplot as plt, warnings
from sklearn.model_selection import train_test_split, cross_val_score
warnings.filterwarnings("ignore")
rng = np.random.default_rng(0)
def check(ok, msg_ok, msg_bad):
    print(("✅ " + msg_ok) if ok else ("❌ " + msg_bad))

---
## Challenge 1 – The polynomial that memorised
A polynomial regression fits 15 noisy points from a smooth curve. With degree 3 it generalises well.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error

def make_curve(n, seed):
    r = np.random.default_rng(seed); x = np.sort(r.uniform(-1, 1, n))
    return x[:, None], np.sin(3 * x) + r.normal(0, 0.2, n)
Xtr, ytr = make_curve(15, 1); Xte, yte = make_curve(200, 2)

def fit_poly(degree, model=None):
    m = make_pipeline(PolynomialFeatures(degree), StandardScaler(), model or LinearRegression()).fit(Xtr, ytr)
    return m, mean_squared_error(ytr, m.predict(Xtr)), mean_squared_error(yte, m.predict(Xte))

m, tr, te = fit_poly(3)
print(f"degree 3: train MSE {tr:.3f}, test MSE {te:.3f}")

### 💥 Break it
Change `degree` so the model **memorises** the training data: train MSE below 0.01 **and** test MSE at least 10× the train MSE.

In [ ]:
degree = 3   # ← TODO: change me

m, tr, te = fit_poly(degree)
grid = np.linspace(-1, 1, 400)[:, None]
plt.scatter(Xtr, ytr, label="train"); plt.plot(grid, np.clip(m.predict(grid), -3, 3), "r", label=f"degree {degree}")
plt.plot(grid, np.sin(3 * grid), "k--", alpha=.4, label="truth"); plt.ylim(-3, 3); plt.legend(); plt.show()
print(f"train MSE {tr:.4f}, test MSE {te:.4f}")
check(tr < 0.01 and te > 10 * tr, "Broken! The curve passes through every training point but wiggles wildly in between.",
      "Not overfitting yet – try a much higher degree (how many points are there?).")

### 🛠️ Fix it
Keep your high degree but swap `LinearRegression` for `Ridge(alpha=...)`. Find an `alpha` that brings the test MSE below 0.1.

In [ ]:
alpha = 1e-8   # ← TODO: change me

m, tr, te = fit_poly(degree, Ridge(alpha=alpha))
print(f"degree {degree}, alpha {alpha}: train MSE {tr:.4f}, test MSE {te:.4f}")
check(degree >= 10 and te < 0.1, "Fixed – regularisation tamed the wiggles.", "Break it first with a high degree, then increase alpha (try powers of 10)." if degree < 10 else "Still overfitting – increase alpha (try powers of 10).")

<details><summary>▶ Solution</summary>

`degree = 14` (with 15 points a degree-14 polynomial can pass through all of them exactly). Then `alpha` around `0.01`–`1` works. **Lesson:** a model with as many parameters as data points can memorise noise; regularisation (or more data) restores generalisation.
</details>

---
## Challenge 2 – 90 % accuracy on pure noise
A researcher has 50 patients and 5,000 random gene measurements – **pure noise**, the labels are coin flips. Any honest method must score about 50 %.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
Xn = rng.normal(size=(50, 5000)); yn = rng.integers(0, 2, 50)
honest = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
print("honest CV accuracy:", cross_val_score(honest, Xn, yn, cv=5).mean().round(3))

### 💥 Break it
Get a cross-validated accuracy **above 0.8** on this noise by doing feature selection the *wrong* way: select the 20 "best" features using **all** the data **before** cross-validation. Fill in the TODO.

In [ ]:
# TODO: create X_selected = the 20 best columns of Xn chosen with SelectKBest(f_classif, k=20) fitted on ALL of Xn, yn
X_selected = Xn[:, :20]   # ← replace this line

leaky = cross_val_score(LogisticRegression(), X_selected, yn, cv=5).mean()
print("leaky CV accuracy:", round(leaky, 3))
check(leaky > 0.8, "Broken! The selector already saw the test folds' labels, so CV 'finds' signal in noise.",
      "Still honest – fit SelectKBest on the full data before calling cross_val_score.")

### 🛠️ Fix it
The `honest` pipeline above is the fix – selection happens **inside** each fold. Explain to yourself in one sentence why it scores ~0.5.

<details><summary>▶ Solution</summary>

```python
X_selected = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)
```
With 5,000 noise features some correlate with the labels **by chance**. Picking them with all labels and then cross-validating means the "test" labels were used to choose the features → optimistic scores. Rule: **every** step that learns from data (scaling, imputation, selection, tuning) goes inside the pipeline/fold.
</details>

---
## Challenge 3 – The suspiciously perfect feature
A bank predicts loan default. The data team added a feature `days_in_collections`.

In [ ]:
n = 2000
income = rng.normal(50, 15, n); debt = rng.normal(20, 8, n)
p_default = 1 / (1 + np.exp(-(debt - 0.4 * income) / 4))
default = rng.random(n) < p_default
days_in_collections = np.where(default, rng.integers(30, 200, n), 0)   # only known AFTER a default!
X_ok = np.c_[income, debt]
print("CV accuracy without the new feature:", cross_val_score(LogisticRegression(), X_ok, default, cv=5).mean().round(3))

### 💥 Break it
Add `days_in_collections` as a third column and reach **≥ 0.99** CV accuracy.

In [ ]:
X_new = X_ok   # ← TODO: add days_in_collections as a column

acc = cross_val_score(LogisticRegression(max_iter=1000), X_new, default, cv=5).mean()
print("CV accuracy:", round(acc, 3))
check(acc >= 0.99, "Broken – a 'perfect' model that is useless in production: the feature only exists after the outcome.",
      "Add the column with np.c_[X_ok, days_in_collections].")

### 🛠️ Fix it
Ask for every feature: **"Would I know this value at the moment I make the prediction?"** If not, drop it. Too-good-to-be-true accuracy is the classic symptom of target leakage.

<details><summary>▶ Solution</summary>

`X_new = np.c_[X_ok, days_in_collections]`. The fix is to remove it (back to `X_ok`). Real-world examples: "treatment given" when predicting a diagnosis, "refund issued" when predicting fraud, final grades when predicting drop-out.
</details>

---
## Challenge 4 – The 99 % accurate fraud detector
Only 1 % of transactions are fraud. A colleague proudly reports 99 % accuracy.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, recall_score, average_precision_score, classification_report
Xf, yf = make_classification(n_samples=20000, n_features=10, n_informative=4, weights=[0.99], class_sep=1.0, flip_y=0, random_state=0)
Xa, Xb, ya, yb = train_test_split(Xf, yf, stratify=yf, random_state=0)
print("fraud rate in test:", yb.mean().round(4))

### 💥 Break it
Write the laziest possible "model": predict **no fraud for everyone**. Show it gets ≥ 98.5 % accuracy while catching **zero** frauds.

In [ ]:
lazy_pred = np.ones_like(yb)   # ← TODO: predict "not fraud" (0) for everyone

acc, rec = accuracy_score(yb, lazy_pred), recall_score(yb, lazy_pred, zero_division=0)
print(f"accuracy {acc:.3f}, recall (frauds caught) {rec:.3f}")
check(acc >= 0.985 and rec == 0, "Broken – accuracy rewards ignoring the rare class entirely.",
      "Predict 0 for every row (np.zeros_like(yb)).")

### 🛠️ Fix it
Train a logistic regression **with** `class_weight="balanced"` and report recall and PR-AUC (average precision) instead of accuracy. Reach recall ≥ 0.6.

In [ ]:
clf = LogisticRegression(max_iter=1000)   # ← TODO: add class_weight="balanced"
clf.fit(Xa, ya)
pred, score = clf.predict(Xb), clf.predict_proba(Xb)[:, 1]
print(classification_report(yb, pred, digits=3, zero_division=0))
print("PR-AUC:", round(average_precision_score(yb, score), 3), "(the lazy model's PR-AUC equals the fraud rate:", yb.mean().round(3), ")")
check(recall_score(yb, pred) >= 0.6, "Fixed – now the model actually looks for fraud (precision drops: that's the trade-off).",
      "Recall still low – use class_weight='balanced' (or lower the decision threshold).")

<details><summary>▶ Solution</summary>

`lazy_pred = np.zeros_like(yb)`; `LogisticRegression(class_weight="balanced", max_iter=1000)`. **Lesson:** on imbalanced data report recall/precision/PR-AUC and compare with the trivial baseline. Accuracy can be high for a useless model.
</details>

---
## Challenge 5 – Fool the digit classifier
A logistic regression reads 8×8 handwritten digits with ~96 % accuracy. Can you change an image **just a little** so it predicts the wrong digit?

For multinomial logistic regression the gradient of the loss w.r.t. the **input** is $\nabla_x L = W^\top(p - y_{\text{onehot}})$. The **fast gradient sign method** (FGSM) moves every pixel a small step $\varepsilon$ in the direction that increases the loss: $x_{adv} = x + \varepsilon\,\mathrm{sign}(\nabla_x L)$.

In [ ]:
from sklearn.datasets import load_digits
dg = load_digits(); Xd = dg.data / 16.0; yd = dg.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, random_state=0)
digit_clf = LogisticRegression(max_iter=2000).fit(Xd_tr, yd_tr)
print("test accuracy:", round(digit_clf.score(Xd_te, yd_te), 3))
x0, y0 = Xd_te[1], yd_te[1]   # try other indices too!
print("true label:", y0, "→ predicted:", digit_clf.predict([x0])[0])

### 💥 Break it
Implement `input_gradient` and find an `eps` (pixel values are in [0, 1]) that flips the prediction. Smaller is more impressive – try to stay below 0.2.

In [ ]:
def input_gradient(clf, x, y):
    p = clf.predict_proba([x])[0]
    onehot = np.eye(len(p))[y]
    # TODO: return W^T (p - onehot), where W = clf.coef_ (shape 10 x 64)
    return np.zeros_like(x)   # ← replace

eps = 0.0   # ← TODO
x_adv = np.clip(x0 + eps * np.sign(input_gradient(digit_clf, x0, y0)), 0, 1)
pred_adv = digit_clf.predict([x_adv])[0]

fig, ax = plt.subplots(1, 2, figsize=(5, 2.6))
ax[0].imshow(x0.reshape(8, 8), cmap="gray_r"); ax[0].set_title(f"original → {digit_clf.predict([x0])[0]}")
ax[1].imshow(x_adv.reshape(8, 8), cmap="gray_r"); ax[1].set_title(f"eps={eps} → {pred_adv}")
for a in ax: a.axis("off")
plt.show()
check(pred_adv != y0, f"Broken! A {eps}-sized nudge per pixel turned a {y0} into a {pred_adv}.",
      "Prediction unchanged – implement the gradient and increase eps.")

### 🛠️ Fix it (discussion)
Defences: **adversarial training** (add `x_adv` examples with their true labels to the training set), input smoothing/quantisation, or models with larger margins. Try it: generate FGSM examples for the training set, retrain on the union, and check whether the same `eps` still fools the new model.

<details><summary>▶ Solution</summary>

```python
return clf.coef_.T @ (p - onehot)
```
For this image `eps = 0.1` is enough; across test images the needed `eps` is typically 0.05–0.25, while the digit still looks the same to a human. Deep networks are even more vulnerable – this is why adversarial robustness is a research field.
</details>

---
## Challenge 6 – It worked in the lab…
A model predicting house prices is trained on small houses only. Production starts selling big houses.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
size = rng.uniform(40, 300, 3000)
price = 3000 * size + 20 * (size - 150) ** 2 + rng.normal(0, 2e4, 3000)
lab = size < 150
rf = RandomForestRegressor(n_estimators=100, random_state=0).fit(size[lab][:, None], price[lab])
mae = lambda m, mask: np.abs(m.predict(size[mask][:, None]) - price[mask]).mean()
print(f"MAE on small houses (lab): {mae(rf, lab):,.0f}")

### 💥 Break it
Define `prod_mask` so it selects the **biggest** houses and show the error explodes (≥ 5× the lab error).

In [ ]:
prod_mask = lab   # ← TODO: select houses with size > 200

err = mae(rf, prod_mask)
grid = np.linspace(40, 300, 300)[:, None]
plt.scatter(size, price, s=3, alpha=.3, label="all houses"); plt.plot(grid, rf.predict(grid), "r", label="forest trained on <150 m²")
plt.axvline(150, ls="--", c="k"); plt.legend(); plt.xlabel("m²"); plt.show()
print(f"MAE in production: {err:,.0f}")
check(err >= 5 * mae(rf, lab), "Broken – trees can't extrapolate: the prediction is flat beyond the training range.",
      "Select size > 200 for production.")

### 🛠️ Fix it
Options: collect training data covering the production range; use a model that extrapolates sensibly (e.g. linear terms); and **monitor input drift** (PSI on `size`) so you notice before customers do.

<details><summary>▶ Solution</summary>

`prod_mask = size > 200`. Try `make_pipeline(PolynomialFeatures(2), LinearRegression())` on the lab data – it extrapolates far better here because the true relationship is quadratic. In general: know your training distribution and alert when production leaves it.
</details>

---
## Challenge 7 – The scale that broke kNN
kNN classifies wines from 13 chemical measurements.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
Xw, yw = load_wine(return_X_y=True)
scaled_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(5))
print("kNN with scaling:", cross_val_score(scaled_knn, Xw, yw, cv=5).mean().round(3))

### 💥 Break it
Remove the scaler (use a plain `KNeighborsClassifier(5)`) and get accuracy **below 0.75**. Then look at `Xw.std(axis=0)` – which feature dominates the distances?

In [ ]:
broken_knn = scaled_knn   # ← TODO: kNN without StandardScaler

acc = cross_val_score(broken_knn, Xw, yw, cv=5).mean()
print("accuracy:", round(acc, 3)); print("feature std devs:", Xw.std(axis=0).round(1))
check(acc < 0.75, "Broken – 'proline' (std ≈ 315) swamps every other feature in the Euclidean distance.",
      "Use KNeighborsClassifier(5) on its own.")

<details><summary>▶ Solution</summary>

`broken_knn = KNeighborsClassifier(5)` → about 0.69. The fix is the original pipeline. Distance-based methods (kNN, k-means, SVM with RBF, PCA) need features on comparable scales; tree models do not.
</details>

---
🎉 **Done!** Which break surprised you most? Write one line about it in your notes.